# project_3_pr_danger.ipynb

- 목적: 원본 기상 특예보 CSV에서 앱이 바로 쓸 수 있는 재난 특보 정리본을 만든다.
- 입력: `danger.csv`
- 출력: `danger_clean.csv`
- 메모: 지역명 통일, 특보/등급 추출, 재난 종류 정규화가 한 셀 안에서 이어지는 노트북이다.


## 처리 흐름
1. `danger.csv` 로딩
2. 결측 제거와 발표시간 변환
3. 지역명 통일과 분리
4. 특보/등급/재난종류 추출
5. `danger_clean.csv` 저장


In [ ]:
import pandas as pd
import re

# 1. danger 파일 불러오기
df = pd.read_csv("data/danger.csv", encoding="cp949")

# 2. 내용 없는 행 제거
df = df.dropna(subset=["해당지역"])

# 3. 발표시간 datetime 변환
df["발표시간"] = pd.to_datetime(df["발표시간"], errors="coerce")

# 4. 지역 이름 통일
df["지역"] = df["지역"].str.replace("경상남도", "경남")
df["지역"] = df["지역"].str.replace("경상북도", "경북")
df["지역"] = df["지역"].str.replace("대구광역시", "대구")
df["지역"] = df["지역"].str.replace("부산광역시", "부산")
df["지역"] = df["지역"].str.replace("울산광역시", "울산")

# 5. 지역 분리 (부산·울산·경남 → 행 분리)
df["지역"] = df["지역"].str.split("·")
df = df.explode("지역")
df["지역"] = df["지역"].str.strip()

# 6. 특보 추출
df["특보"] = df["해당지역"].str.extract(
"(강풍주의보|강풍경보|호우주의보|호우경보|풍랑주의보|풍랑경보|대설주의보|대설경보|한파주의보|한파경보|폭염주의보|폭염경보|태풍주의보|태풍경보|건조주의보|건조경보)"
)

# 7. 특보 등급
df["특보등급"] = df["특보"].str.extract("(주의보|경보)")

# 8. 재난 종류
df["재난종류"] = df["특보"].str.replace("주의보|경보", "", regex=True)

# 9. 시군구 추출 함수
def extract_sigungu(text):

    if pd.isna(text):
        return []

    results = []

    # 괄호 안 지역 추출
    for g in re.findall(r"\(([^()]*)\)", text):
        parts = re.split(r"[.\s]+", g)

        for p in parts:
            p = p.strip()

            if not p:
                continue
            if p.isdigit():
                continue

            results.append(p)

    # 괄호 없는 경우 (울릉도.독도 같은 형태)
    if not results and ":" in str(text):
        tail = text.split(":")[-1]
        parts = re.split(r"[.]", tail)

        results += [p.strip() for p in parts if p.strip()]

    return results


# 10. 시군구 생성
df["시군구"] = df["해당지역"].apply(extract_sigungu)

# 11. 행 분리
df = df.explode("시군구")

# 12. 필요 없는 컬럼 제거
df = df.drop(columns=["발효시각"], errors="ignore")

# 13. 컬럼 정리
df = df[
[
"발표시간",
"지역",
"시군구",
"재난종류",
"특보등급",
"해당지역"
]
]

# 14. 중복 제거
df = df.drop_duplicates()

# 15. 저장
df.to_csv(
"data/danger_clean.csv",
index=False,
encoding="utf-8-sig"
)

print("전처리 완료")
print("데이터 수:", len(df))


## 결과 메모
이 노트북은 기상 특예보 문장에서 추천 페이지가 쓸 `지역`, `시군구`, `재난종류`, `특보등급`을 뽑아내는 전처리 단계다.
